# REA Colab Quickstart

Run one or more repository samplers from a fresh Colab GPU runtime. The default selection runs only a substantial end-to-end masked Barker SMC experiment, followed by validation tables and diagnostic plots.

In [ ]:
# EDIT PARAMETERS HERE
ALGORITHMS_TO_RUN = [
    "masked_barker_smc",
]

SEED = 12345
NUM_PARTICLES = 64
MAX_NEW_TOKENS = 100
LAMBDAS = "0,-0.25,-0.5,-1.0,-1.5,-2.0"
MCMC_STEPS_PER_LEVEL = 5
ESS_THRESHOLD = 0.5
MASK_MODE = "stochastic"
MASK_SIZES = "1,2,4,8"
MASK_RHO_SCHEDULE = "0.4,0.3,0.2,0.1"
MASK_POLICY = "uniform_noncontiguous"
PROPOSAL_TEMPERATURE = 1.0
BATCH_SIZE = 32
EXPERIMENT_LABEL = "masked_barker_smc_substantial"

# Existing pipelines are opt-in: add "full_sequence_smc" or "mixture_mcmc".
VALID_ALGORITHMS = {"masked_barker_smc", "full_sequence_smc", "mixture_mcmc"}
unknown = set(ALGORITHMS_TO_RUN) - VALID_ALGORITHMS
if unknown:
    raise ValueError(f"Unknown algorithms: {sorted(unknown)}")
print("Selected:", ALGORITHMS_TO_RUN)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git"
REPO_DIR = pathlib.Path("/content/rea_of_llms_supplementary_code-main")
OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/rea_outputs")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("cwd =", os.getcwd())
print("outputs =", OUTPUT_ROOT)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import torch
print("torch:", torch.__version__)
print("compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Run selected pipelines

Each selected algorithm writes to its own Drive directory. With the default configuration, neither the older SMC nor mixture MCMC is imported or executed.

In [ ]:
commands = {}
output_dirs = {
    "masked_barker_smc": OUTPUT_ROOT / EXPERIMENT_LABEL,
    "full_sequence_smc": OUTPUT_ROOT / "full_sequence_smc",
    "mixture_mcmc": OUTPUT_ROOT / "mixture_mcmc",
}

masked_output = output_dirs["masked_barker_smc"]
commands["masked_barker_smc"] = [
    "python", "masked_barker_smc_experiment/run_experiment.py",
    "--device", DEVICE,
    "--num-particles", str(NUM_PARTICLES),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--lambdas", LAMBDAS,
    "--mcmc-steps-per-level", str(MCMC_STEPS_PER_LEVEL),
    "--ess-threshold", str(ESS_THRESHOLD),
    "--mask-mode", MASK_MODE,
    "--mask-sizes", MASK_SIZES,
    "--mask-rho-schedule", MASK_RHO_SCHEDULE,
    "--mask-policy", MASK_POLICY,
    "--proposal-temperature", str(PROPOSAL_TEMPERATURE),
    "--batch-size", str(BATCH_SIZE),
    "--seed", str(SEED),
    "--run-name", EXPERIMENT_LABEL,
    "--output-dir", str(masked_output),
]

commands["full_sequence_smc"] = [
    "python", "full_sequence_smc_experiment/run_smc_sampler.py",
    "--device", DEVICE,
    "--num-particles", str(NUM_PARTICLES),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--lambdas", LAMBDAS,
    "--mcmc-steps-per-level", str(MCMC_STEPS_PER_LEVEL),
    "--seed", str(SEED),
    "--output-dir", str(output_dirs["full_sequence_smc"]),
]

commands["mixture_mcmc"] = [
    "python", "mixture_mcmc_experiment/run_experiment.py",
    "--device", DEVICE,
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--seed", str(SEED),
    "--output-dir", str(output_dirs["mixture_mcmc"]),
]

for algorithm in ALGORITHMS_TO_RUN:
    print(f"\nRunning {algorithm} ...", flush=True)
    subprocess.run(commands[algorithm], check=True)

In [ ]:
from IPython.display import Markdown, display

for algorithm in ALGORITHMS_TO_RUN:
    report_path = output_dirs[algorithm] / "report.md"
    summary_path = output_dirs[algorithm] / "summary.json"
    if not report_path.exists():
        print(f"No report found for {algorithm}: {report_path}")
        continue
    display(Markdown(report_path.read_text(encoding="utf-8")))
    print("Summary:", summary_path)

## Validate the substantial masked Barker SMC run

The next cells verify the complete particle output, display the main metrics and level schedule, plot the sampler diagnostics, and show the highest-ARI completions. They do not launch another simulation.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

if "masked_barker_smc" not in ALGORITHMS_TO_RUN:
    raise RuntimeError("Add 'masked_barker_smc' to ALGORITHMS_TO_RUN and execute the run cell first.")

result_dir = output_dirs["masked_barker_smc"]
summary_path = result_dir / "summary.json"
samples_path = result_dir / "samples.jsonl"
if not summary_path.exists() or not samples_path.exists():
    raise FileNotFoundError("Run the selected-pipelines cell before validating the experiment.")

payload = json.loads(summary_path.read_text(encoding="utf-8"))
samples_df = pd.read_json(samples_path, lines=True)
levels_df = pd.DataFrame(payload["diagnostics"]["level_diagnostics"])
weights = samples_df["normalized_weight"].to_numpy(dtype=float)

# End-to-end integrity checks.
assert payload["algorithm"] == "masked_barker_smc"
assert payload["config"]["lambdas"] == [float(x) for x in LAMBDAS.split(",")]
assert len(samples_df) == NUM_PARTICLES
assert samples_df["completion_token_ids"].map(len).eq(MAX_NEW_TOKENS).all()
assert np.isfinite(weights).all() and np.isclose(weights.sum(), 1.0, atol=1e-6)
assert np.isfinite(samples_df["raw_ari"].to_numpy(dtype=float)).all()
assert len(levels_df) == len(LAMBDAS.split(","))
assert 0.0 < payload["final_ess_over_n"] <= 1.0
print("End-to-end checks passed.")

metrics = pd.DataFrame([
    {
        "particles": payload["num_particles"],
        "completion_tokens": payload["max_new_tokens"],
        "target_lambda": payload["target_lambda"],
        "runtime_seconds": payload["runtime_total_seconds"],
        "seconds_per_particle": payload["runtime_seconds_per_final_particle"],
        "final_ess_over_n": payload["final_ess_over_n"],
        "min_ess_over_n": payload["min_ess_over_n"],
        "acceptance_rate": payload["mean_mcmc_acceptance_rate"],
        "resampling_steps": payload["resampling_steps"],
        "distinct_root_ancestors": payload["final_distinct_root_ancestors"],
        "mean_raw_ari": payload["mean_ari"],
        "min_raw_ari": payload["min_ari"],
        "max_raw_ari": payload["max_ari"],
    }
])
display(metrics)
display(levels_df[
    ["level", "lambda", "mutation_target_lambda", "ess_over_n_before_resampling",
     "ess_over_n_after_resampling", "resampled", "acceptance_rate",
     "mean_symmetrization_probability", "mean_barker_probability", "mean_ari"]
])

tail_table = pd.DataFrame([
    {"threshold": key.replace("ari_ge_", "ARI >= "), "count": count,
     "proportion": payload["tail_proportions"][key]}
    for key, count in payload["tail_counts"].items()
])
display(tail_table)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
lambdas_plot = levels_df["lambda"].to_numpy(dtype=float)

axes[0, 0].plot(lambdas_plot, levels_df["mean_ari"], marker="o", color="tab:green")
axes[0, 0].set(title="Mean ARI along the SMC schedule", xlabel="lambda", ylabel="mean capped ARI")
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(lambdas_plot, levels_df["ess_over_n_before_resampling"], marker="o", label="before resampling")
axes[0, 1].plot(lambdas_plot, levels_df["ess_over_n_after_resampling"], marker="s", label="after resampling")
axes[0, 1].axhline(ESS_THRESHOLD, color="tab:red", linestyle="--", label="ESS threshold")
axes[0, 1].set(title="Particle degeneracy", xlabel="lambda", ylabel="ESS / N", ylim=(0, 1.05))
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

mutation_levels = levels_df[levels_df["level"] > 0]
axes[1, 0].plot(mutation_levels["lambda"], mutation_levels["acceptance_rate"], marker="o", label="total acceptance")
axes[1, 0].plot(mutation_levels["lambda"], mutation_levels["mean_symmetrization_probability"], marker="s", label="mean symmetrization factor")
axes[1, 0].plot(mutation_levels["lambda"], mutation_levels["mean_barker_probability"], marker="^", label="mean Barker factor")
axes[1, 0].set(title="Mutation diagnostics", xlabel="new lambda after weight update", ylabel="probability", ylim=(0, 1.05))
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

raw_ari = samples_df["raw_ari"].to_numpy(dtype=float)
axes[1, 1].hist(raw_ari, bins="auto", color="tab:blue", alpha=0.8, edgecolor="white")
for threshold in (6, 7, 8):
    axes[1, 1].axvline(threshold, linestyle="--", linewidth=1, label=f"ARI >= {threshold}")
axes[1, 1].set(title="Final raw ARI distribution", xlabel="raw ARI", ylabel="final particle count")
axes[1, 1].legend()

fig.tight_layout()
plt.show()

In [ ]:
top_completions = samples_df.sort_values("raw_ari", ascending=False).drop_duplicates("completion").head(10)
display(top_completions[["particle_id", "raw_ari", "capped_ari", "normalized_weight", "completion"]])

## Optional selected-result publication

A normal run writes only to Drive. To add a reviewed run to the repository, rerun its command with `--experiment-log results/EXPERIMENT_LOG.md --publish-result-dir results/selected_runs`, inspect the small generated files, then commit them intentionally.